In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import tensorflow as tf

In [ ]:
df = pd.read_csv("C:/Users/marck/Downloads/nahr_ibrahim_watershed/data/master/nahr_ibrahim_master_full.csv", parse_dates=["date"])

# Monthly averages — check seasonal patterns
monthly = df.groupby("month").mean(numeric_only=True)

fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

axes[0].bar(monthly.index, monthly["precip_mm_day"], color="steelblue")
axes[0].set_title("Monthly Average Precipitation (mm/day)")

axes[1].plot(monthly.index, monthly["temp_mean_c"], color="tomato", marker="o")
axes[1].set_title("Monthly Average Temperature (°C)")

axes[2].bar(monthly.index, monthly["snow_cover_pct"], color="skyblue")
axes[2].set_title("Monthly Average Snow Cover (%)")

axes[3].bar(monthly.index, monthly["discharge_m3s"], color="seagreen")
axes[3].set_title("Monthly Average Discharge (m³/s)")

for ax in axes:
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                         "Jul","Aug","Sep","Oct","Nov","Dec"])
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("C:/Users/marck/Downloads/nahr_ibrahim_watershed/results/figures/seasonal_overview.png", dpi=150)
plt.show()

In [ ]:
SEQ_DIR = Path("C:/Users/marck/Downloads/nahr_ibrahim_watershed/data/sequences")

X_train = np.load(SEQ_DIR / "X_train.npy")

# Replace the single NaN with 0 (swe_delta of 0 = no snowmelt change)
nan_mask = np.isnan(X_train)
print(f"NaN count before: {nan_mask.sum()}")
X_train[nan_mask] = 0.0
print(f"NaN count after:  {np.isnan(X_train).sum()}")

np.save(SEQ_DIR / "X_train.npy", X_train)
print("X_train.npy updated.")

In [ ]:
MODEL_DIR = Path("C:/Users/marck/Downloads/nahr_ibrahim_watershed/models/trained")

expected = {
    "LSTM"        : "lstm_final.keras",
    "CNN-LSTM"    : "cnn_lstm_final.keras",
    "Transformer" : "transformer_final.keras",
    "TFT"         : "tft_final.keras",
}

print("Model files check:")
for name, fname in expected.items():
    path = MODEL_DIR / fname
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"  {status} {name:<14} → {fname}")

# Also check CMIP6 data
CMIP6_DIR = Path("C:/Users/marck/Downloads/nahr_ibrahim_watershed/data/raw/cmip6")
print("\nCMIP6 data check:")
for scenario in ["ssp245", "ssp585"]:
    for variable in ["pr", "tas", "tasmin", "tasmax"]:
        folder = CMIP6_DIR / scenario / variable
        count  = len(list(folder.glob("*.nc"))) if folder.exists() else 0
        status = "✅" if count == 86 else f"⚠ {count}/86"
        print(f"  {status} {scenario}/{variable}")

In [ ]:
ckpt_dir = Path("C:/Users/marck/Downloads/nahr_ibrahim_watershed/models/checkpoints")
for f in ckpt_dir.glob("*.keras"):
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

In [ ]:
from meteostat import Point, Daily, Stations
from datetime import datetime

stations = Stations()
stations = stations.nearby(34.093, 35.878)
nearby   = stations.fetch(10)
print(nearby[["name","country","elevation","distance"]].to_string())